In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# ── 1. CREATE DATASET ──────────────────────────────────────
np.random.seed(42)
n = 500

df = pd.DataFrame({
    'brand':        np.random.choice(['Toyota','Honda','BMW','Ford','Hyundai'], n),
    'year':         np.random.randint(2010, 2024, n),
    'km_driven':    np.random.randint(5000, 200000, n),
    'fuel':         np.random.choice(['Petrol','Diesel','Electric'], n),
    'transmission': np.random.choice(['Manual','Automatic'], n),
    'max_power':    np.random.randint(70, 300, n),
})

# Realistic price formula
brand_val = {'Toyota':1.1,'Honda':1.0,'BMW':2.5,'Ford':0.9,'Hyundai':0.95}
fuel_val  = {'Petrol':1.0,'Diesel':1.1,'Electric':1.4}

df['price'] = (
    300000
    * df['brand'].map(brand_val)
    * df['fuel'].map(fuel_val)
    * (1 + (df['year'] - 2010) * 0.05)
    * (1 - df['km_driven'] / 400000)
    * (1 + df['max_power'] / 250)
    + np.random.normal(0, 30000, n)
).clip(100000, 5000000).round(-3)

print("Dataset shape:", df.shape)
print(df.head())

# ── 2. PREPROCESSING ───────────────────────────────────────
le = LabelEncoder()
for col in ['brand', 'fuel', 'transmission']:
    df[col] = le.fit_transform(df[col])

df['car_age'] = 2024 - df['year']
df.drop(columns=['year'], inplace=True)

# ── 3. TRAIN / TEST SPLIT ──────────────────────────────────
X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 4. TRAIN MODEL ─────────────────────────────────────────
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ── 5. EVALUATE ────────────────────────────────────────────
y_pred = model.predict(X_test)
print(f"\nR² Score : {r2_score(y_test, y_pred):.4f}")
print(f"MAE      : ₹{mean_absolute_error(y_test, y_pred):,.0f}")

# ── 6. PREDICT A NEW CAR ───────────────────────────────────
new_car = pd.DataFrame([{
    'brand': 2,           # Toyota (encoded)
    'km_driven': 40000,
    'fuel': 1,            # Petrol (encoded)
    'transmission': 0,    # Manual (encoded)
    'max_power': 90,
    'car_age': 5
}])

predicted = model.predict(new_car)[0]
print(f"\nPredicted Price: ₹{predicted:,.0f}")

Dataset shape: (500, 7)
     brand  year  km_driven      fuel transmission  max_power      price
0     Ford  2023      93236  Electric    Automatic        205   880000.0
1  Hyundai  2013     102579    Diesel       Manual        166   473000.0
2      BMW  2018     142541  Electric    Automatic        192  1654000.0
3  Hyundai  2015     166177    Petrol    Automatic        139   338000.0
4  Hyundai  2022      47101  Electric       Manual        238  1092000.0

R² Score : 0.8941
MAE      : ₹90,833

Predicted Price: ₹680,810
